# Auditoria e Integração dos Dados — Tech Challenge Fase 3

Este notebook documenta a auditoria da camada Gold construída na Fase 2 e a preparação dos dados necessários para o desenvolvimento dos modelos de Machine Learning da Fase 3.

Os principais objetivos desta etapa são:

- validar a estrutura, granularidade e qualidade das tabelas Gold;
- identificar limitações e lacunas herdadas da Fase 2;
- incorporar os dados individuais de alunos;
- definir a população válida para modelagem;
- validar as chaves de integração entre alunos e dados agregados;
- preparar uma base analítica reproduzível para a análise exploratória e modelagem.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
GOLD_DIR = PROJECT_ROOT / "data" / "gold"

gold_files = sorted(GOLD_DIR.glob("*.parquet"))

print("Diretório Gold:")
print(GOLD_DIR)

print("\nTabelas encontradas:")
for file in gold_files:
    print("-", file.name)

Diretório Gold:
/home/vitoria/pipeline-alfabetizacao-brasil/data/gold

Tabelas encontradas:
- comparativo_metas_municipio_brasil.parquet
- comparativo_metas_uf_brasil.parquet
- indicadores_alfabetizacao_municipio.parquet
- indicadores_alfabetizacao_uf.parquet


In [2]:
import pandas as pd

gold_tables = {}

for file in gold_files:
    table_name = file.stem
    df = pd.read_parquet(file)
    gold_tables[table_name] = df

    print(f"\n{'=' * 70}")
    print(table_name)
    print(f"Shape: {df.shape}")
    print("Colunas:")
    for column in df.columns:
        print(f"- {column}")


comparativo_metas_municipio_brasil
Shape: (10704, 13)
Colunas:
- ano
- id_municipio
- rede_descricao
- taxa_alfabetizacao
- meta_referencia_ano
- distancia_para_meta_municipio
- status_meta_municipio
- taxa_alfabetizacao_brasil
- meta_referencia_brasil
- diferenca_taxa_municipio_brasil
- nivel_alfabetizacao
- percentual_participacao
- percentual_participacao_brasil

comparativo_metas_uf_brasil
Shape: (54, 12)
Colunas:
- ano
- sigla_uf
- rede_descricao
- taxa_alfabetizacao
- meta_referencia_ano
- distancia_para_meta_uf
- status_meta_uf
- taxa_alfabetizacao_brasil
- meta_referencia_brasil
- diferenca_taxa_uf_brasil
- percentual_participacao
- percentual_participacao_brasil

indicadores_alfabetizacao_municipio
Shape: (23995, 8)
Colunas:
- ano
- id_municipio
- serie_codigo
- rede_codigo
- taxa_alfabetizacao
- media_portugues
- distancia_ponto_corte
- status_ponto_corte

indicadores_alfabetizacao_uf
Shape: (145, 8)
Colunas:
- ano
- sigla_uf
- serie_codigo
- rede_codigo
- taxa_alfabetizacao

In [3]:
gold_tables["indicadores_alfabetizacao_municipio"]

,ano,id_municipio,serie_codigo,rede_codigo,taxa_alfabetizacao,media_portugues,distancia_ponto_corte,status_ponto_corte
0,2023,1100031,2,3,69.10,767.8763,24.8763,acima_ou_igual_ao_ponto_corte
1,2023,1100072,2,3,58.20,747.8918,4.8918,acima_ou_igual_ao_ponto_corte
2,2023,1100189,2,5,69.73,762.4062,19.4062,acima_ou_igual_ao_ponto_corte
3,2023,1101609,2,3,50.70,745.6802,2.6802,acima_ou_igual_ao_ponto_corte
4,2023,1101807,2,3,55.69,752.3724,9.3724,acima_ou_igual_ao_ponto_corte
...,...,...,...,...,...,...,...,...
23990,2024,4319364,2,3,90.00,793.4553,50.4553,acima_ou_igual_ao_ponto_corte
23991,2024,4319364,2,5,90.00,793.4553,50.4553,acima_ou_igual_ao_ponto_corte
23992,2024,3146008,2,2,100.00,792.3100,49.3100,acima_ou_igual_ao_ponto_corte
23993,2024,3529104,2,3,100.00,804.6015,61.6015,acima_ou_igual_ao_ponto_corte


## Inventário da camada Gold

A camada Gold herdada da Fase 2 contém quatro datasets analíticos:

- **`indicadores_alfabetizacao_municipio`**: indicadores agregados por ano, município, série e rede de ensino, incluindo taxa de alfabetização, média de proficiência em Língua Portuguesa e distância em relação ao ponto de corte de 743 pontos.

- **`indicadores_alfabetizacao_uf`**: versão estadual dos indicadores de alfabetização, agregada por ano, UF, série e rede.

- **`comparativo_metas_municipio_brasil`**: reúne os resultados e metas municipais e acrescenta referências nacionais para comparação do desempenho dos municípios com o Brasil.

- **`comparativo_metas_uf_brasil`**: reúne resultados e metas estaduais e permite comparação das UFs com a referência nacional.

Essas tabelas possuem granularidade agregada e, portanto, não representam observações individuais de alunos. Essa limitação é relevante para a Fase 3, cujo problema de Machine Learning exige uma variável-alvo no nível do estudante.

In [4]:
# Definindo as chaves que representam a granularidade de cada tabela
audit_config = {
    "indicadores_alfabetizacao_municipio": [
        "ano",
        "id_municipio",
        "serie_codigo",
        "rede_codigo",
    ],
    "indicadores_alfabetizacao_uf": [
        "ano",
        "sigla_uf",
        "serie_codigo",
        "rede_codigo",
    ],
    "comparativo_metas_municipio_brasil": [
        "ano",
        "id_municipio",
        "rede_descricao",
    ],
    "comparativo_metas_uf_brasil": [
        "ano",
        "sigla_uf",
        "rede_descricao",
    ],
}

audit_results = []

# Auditando a estrutura e a qualidade básica das tabelas Gold
for table_name, key_columns in audit_config.items():
    df = gold_tables[table_name]

    audit_results.append(
        {
            "tabela": table_name,
            "linhas": len(df),
            "colunas": df.shape[1],
            "ano_min": df["ano"].min(),
            "ano_max": df["ano"].max(),
            "nulos_total": int(df.isna().sum().sum()),
            "duplicados_chave": int(
                df.duplicated(subset=key_columns).sum()
            ),
        }
    )

audit_summary = pd.DataFrame(audit_results)

audit_summary

,tabela,linhas,colunas,ano_min,ano_max,nulos_total,duplicados_chave
0,indicadores_alfabetizacao_municipio,23995,8,2023,2024,0,0
1,indicadores_alfabetizacao_uf,145,8,2023,2024,0,0
2,comparativo_metas_municipio_brasil,10704,13,2023,2024,16776,0
3,comparativo_metas_uf_brasil,54,12,2023,2024,99,0


In [5]:
# Detalhando os valores ausentes nas tabelas de comparação
comparison_tables = [
    "comparativo_metas_municipio_brasil",
    "comparativo_metas_uf_brasil",
]

for table_name in comparison_tables:
    df = gold_tables[table_name]

    null_summary = (
        df.isna()
        .sum()
        .loc[lambda x: x > 0]
        .sort_values(ascending=False)
    )

    print(f"\n{'=' * 70}")
    print(table_name)
    print(null_summary)


comparativo_metas_municipio_brasil
meta_referencia_ano                5472
distancia_para_meta_municipio      5472
meta_referencia_brasil             5352
taxa_alfabetizacao                  120
diferenca_taxa_municipio_brasil     120
nivel_alfabetizacao                 120
percentual_participacao             120
dtype: int64

comparativo_metas_uf_brasil
meta_referencia_ano         30
distancia_para_meta_uf      30
meta_referencia_brasil      27
taxa_alfabetizacao           4
diferenca_taxa_uf_brasil     4
percentual_participacao      4
dtype: int64


In [7]:
# Verificando como os valores ausentes se distribuem por ano
for table_name in comparison_tables:
    df = gold_tables[table_name]

    print(f"\n{'=' * 70}")
    print(table_name)

    value_columns = [
        column for column in df.columns
        if column != "ano"
    ]

    nulls_by_year = (
        df[value_columns]
        .isna()
        .groupby(df["ano"])
        .sum()
    )

    columns_with_nulls = nulls_by_year.columns[
        nulls_by_year.sum() > 0
    ]

    print(nulls_by_year[columns_with_nulls])


comparativo_metas_municipio_brasil
      taxa_alfabetizacao  meta_referencia_ano  distancia_para_meta_municipio  \
ano                                                                            
2023                 120                 5352                           5352   
2024                   0                  120                            120   

      meta_referencia_brasil  diferenca_taxa_municipio_brasil  \
ano                                                             
2023                    5352                              120   
2024                       0                                0   

      nivel_alfabetizacao  percentual_participacao  
ano                                                 
2023                  120                      120  
2024                    0                        0  

comparativo_metas_uf_brasil
      taxa_alfabetizacao  meta_referencia_ano  distancia_para_meta_uf  \
ano                                                                 

In [8]:
municipio_comparativo = gold_tables["comparativo_metas_municipio_brasil"]

municipios_sem_meta_2024 = municipio_comparativo[
    (municipio_comparativo["ano"] == 2024)
    & (municipio_comparativo["meta_referencia_ano"].isna())
][
    [
        "id_municipio",
        "rede_descricao",
        "taxa_alfabetizacao",
        "meta_referencia_ano",
        "status_meta_municipio",
        "nivel_alfabetizacao",
        "percentual_participacao",
    ]
]

print("Quantidade de municípios sem meta em 2024:")
print(len(municipios_sem_meta_2024))

municipios_sem_meta_2024.head(20)

Quantidade de municípios sem meta em 2024:
120


,id_municipio,rede_descricao,taxa_alfabetizacao,meta_referencia_ano,status_meta_municipio,nivel_alfabetizacao,percentual_participacao
1,4301750,Municipal,4.40,NaN,sem_meta_ou_resultado,0.0,92.59
9,2414456,Municipal,10.95,NaN,sem_meta_ou_resultado,0.0,80.43
12,2406205,Municipal,11.65,NaN,sem_meta_ou_resultado,0.0,74.29
20,2408706,Municipal,14.29,NaN,sem_meta_ou_resultado,0.0,70.00
27,1200435,Municipal,15.62,NaN,sem_meta_ou_resultado,0.0,76.32
59,1301506,Municipal,19.30,NaN,sem_meta_ou_resultado,0.0,74.59
65,1200328,Municipal,19.59,NaN,sem_meta_ou_resultado,0.0,93.88
93,1300839,Municipal,21.72,NaN,sem_meta_ou_resultado,0.0,89.87
109,2928307,Municipal,22.95,NaN,sem_meta_ou_resultado,0.0,82.43
152,1503754,Municipal,24.95,NaN,sem_meta_ou_resultado,0.0,81.30


## Integração dos dados de alunos

A Fase 3 exige uma variável-alvo no nível individual. Por isso, a tabela `alunos`, disponibilizada pela Base dos Dados via BigQuery, foi incorporada ao pipeline.

Os dados foram armazenados na camada Bronze e padronizados na Silver, preservando todos os registros da fonte. Para a modelagem, serão considerados apenas alunos presentes e com a prova preenchida, pois os demais registros não representam avaliações válidas.

A variável `alfabetizado` será utilizada como variável-alvo do problema de classificação.

In [10]:
SILVER_DIR = PROJECT_ROOT / "data" / "silver"

alunos = pd.read_parquet(
    SILVER_DIR / "alunos.parquet",
    columns=[
        "ano",
        "id_municipio",
        "id_aluno",
        "serie_codigo",
        "rede_codigo",
        "presenca_codigo",
        "preenchimento_caderno_codigo",
        "alfabetizado",
        "proficiencia",
        "peso_aluno",
    ],
)

alunos["avaliacao_valida"] = (
    (alunos["presenca_codigo"] == "1")
    & (alunos["preenchimento_caderno_codigo"] == "1")
)

print("Total de registros:", len(alunos))
print("Avaliações válidas:", alunos["avaliacao_valida"].sum())
print("Avaliações não válidas:", (~alunos["avaliacao_valida"]).sum())

print("\nDistribuição do target nas avaliações válidas:")
print(
    alunos.loc[alunos["avaliacao_valida"]]
    .groupby(["ano", "alfabetizado"])
    .size()
)

Total de registros: 3867999
Avaliações válidas: 3354661
Avaliações não válidas: 513338

Distribuição do target nas avaliações válidas:
ano   alfabetizado
2023  0                625382
      1                877427
2024  0                744733
      1               1107119
dtype: int64


## População utilizada na modelagem

A base possui 3.867.999 registros de alunos. Para as próximas etapas, foram mantidos apenas os 3.354.661 alunos presentes e com a prova preenchida, pois somente esses registros representam avaliações válidas.

Os 513.338 registros restantes foram preservados nas camadas Bronze e Silver, mas não fazem parte da população utilizada para prever alfabetização.

A variável `alfabetizado` será utilizada como target. A variável `proficiencia` poderá ser utilizada nas análises exploratórias e para validação dos dados, mas não será utilizada como variável explicativa do modelo para evitar vazamento de informação.

A população válida foi salva na Gold como `alunos_avaliacoes_validas.parquet`.